In [ ]:

import pandas as pd

def read_and_clean_data(file_path, sheet_name = 1):  #copied from algorithm.ipynb
    df = pd.read_excel(file_path, sheet_name = sheet_name)

    #df = df.iloc[1: , 1:]

    df = df.fillna(0)

    #print(df)
    return df


def create_coverage_dictionary(df):  #this is also almost the same as in algorithm.ipynb, but now we want to return a dictionary of sets instead of lists, to make it easier to check if a demand point is covered by a station
    # in algorithm.ipynb the dictionary we create here is called dictionary 1 

    coverage = {}

    travel_time = {} 

    n_rows, n_cols = df.shape

    for i in range(1, n_rows):

        row_key = int(df.iloc[i, 0])

        coverage[row_key] = set()

        for j in range(1, n_cols):

            travel_time[(row_key, int(df.iloc[0, j]))] = df.iloc[i, j]

            if df.iloc[i, j] <= 900:

                coverage[row_key].add(int(df.iloc[0, j]))

                
    return coverage, travel_time

def create_demand_dictionary(file_path, sheet_name = 2):
    demand_df = read_and_clean_data(file_path, sheet_name = sheet_name)
    
    demand_dictionary = {}
    
    for i in range(0, len(demand_df)):
        postal_code = int(demand_df.iloc[i, 0])
        demand = float(demand_df.iloc[i, 1])
        demand_dictionary[postal_code] = demand
    
    return demand_dictionary



file = "Data assignment ambulance 2 Large.xlsx"

travel_df = read_and_clean_data(file, sheet_name = 1)

# Create coverage dictionary and travel time dictionary
coverage, travel_time = create_coverage_dictionary(travel_df)

# Create demand dictionary
demand = create_demand_dictionary(file, sheet_name = 2)

# just a list of all demand locations
locations = list(coverage.keys())

print(locations)





[4251, 4254, 4255, 4261, 4264, 4265, 4266, 4267, 4268, 4269, 4271, 4273, 4281, 4283, 4284, 4285, 4286, 4287, 4288, 4611, 4612, 4613, 4614, 4615, 4616, 4617, 4621, 4622, 4623, 4624, 4625, 4631, 4634, 4635, 4641, 4645, 4651, 4652, 4655, 4661, 4664, 4671, 4681, 4701, 4702, 4703, 4704, 4705, 4706, 4707, 4708, 4709, 4711, 4714, 4715, 4721, 4722, 4724, 4725, 4726, 4727, 4731, 4735, 4741, 4744, 4751, 4754, 4756, 4758, 4761, 4762, 4765, 4766, 4771, 4772, 4781, 4782, 4791, 4793, 4794, 4796, 4797, 4811, 4812, 4813, 4814, 4815, 4816, 4817, 4818, 4819, 4822, 4823, 4824, 4825, 4826, 4827, 4834, 4835, 4836, 4837, 4838, 4839, 4841, 4844, 4845, 4847, 4849, 4851, 4854, 4855, 4856, 4858, 4859, 4861, 4871, 4872, 4873, 4874, 4875, 4876, 4877, 4878, 4879, 4881, 4882, 4884, 4885, 4891, 4901, 4902, 4903, 4904, 4905, 4906, 4907, 4908, 4909, 4911, 4921, 4924, 4926, 4927, 4931, 4941, 4942, 4944, 5011, 5012, 5013, 5014, 5015, 5017, 5018, 5021, 5022, 5025, 5026, 5032, 5035, 5036, 5037, 5038, 5041, 5042, 5043, 504

In [ ]:
import gurobipy as gp
from gurobipy import GRB
# step 1

# we will build this model in two steps the first one maximizing the covered demand and the second one minimizing avg travel time(while keeping that same covered demand as in step 1 )
I = locations
J = I

P = 5   #this is for our large dataset

# Create a new model
model = gp.Model("ambulance_locations")

# variables

x = model.addVars(J, vtype=GRB.BINARY, name="x")
y = model.addVars(I, vtype=GRB.BINARY, name="y")

#constraint number 1 we must choose P ambulances
model.addConstr(gp.quicksum(x[j] for j in J) == P)

#constraint number 2 we can only cover a demand point if we have an ambulance in a station that can cover it
for i in I:
    model.addConstr(y[i] <= gp.quicksum(x[j] for j in coverage[i]))


# now our objective function we want to maximize total demand

model.setObjective(
    gp.quicksum(demand[i] * y[i] for i in I),
    GRB.MAXIMIZE
)

#we solve it

model.optimize()

#we store it bcs we need to use it later
Z_star = model.ObjVal


print("Maximum covered demand:", Z_star)


chosen_locations_step1 = [j for j in J if x[j].X > 0.5]
covered_locations_step1 = [i for i in I if y[i].X > 0.5]

print("Chosen locations in step 1:", chosen_locations_step1)
print("Number of covered locations:", len(covered_locations_step1))
print("Covered demand:", Z_star)





Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 218 rows, 434 columns and 10773 nonzeros (Max)
Model fingerprint: 0x4f86f89f
Model has 217 linear objective coefficients
Variable types: 0 continuous, 434 integer (434 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+00, 2e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 5e+00]

Found heuristic solution: objective 776825.00000
Presolve removed 4 rows and 8 columns
Presolve time: 0.01s
Presolved: 214 rows, 426 columns, 10549 nonzeros
Variable types: 0 continuous, 426 integer (423 binary)

Root relaxation: objective 1.046665e+06, 156 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     

In [ ]:

# Step 2: We keep the best coverage and then we minimize travel time


#add the Z variable

poss_assignments = []
for i in I:
    for j in coverage[i]: 
        # we also need to only check for j in the coverage of i, because we cannot assign a demand point to a postal code that doesnt cover it.
        poss_assignments.append((j, i))

z = model.addVars(poss_assignments, vtype=GRB.BINARY, name="z")
# for all i the sum over j of z[j, i] must be equal to y[i]
for i in I:
    model.addConstr(gp.quicksum(z[j, i] for j in coverage[i]) == y[i])
# for all i and j, z[j, i] can only be 1 if x[j] is 1, so if we actually have an ambulance in j that can cover i
for i in I:
    for j in coverage[i]:
        model.addConstr(z[j, i] <= x[j])
# we also need to add the constraint that the total covered demand must be equal to Z_star, because we want to keep the same coverage as in step 1
model.addConstr(gp.quicksum(demand[i] * y[i] for i in I) == Z_star)

#this is the second objective function
model.setObjective(
    gp.quicksum(demand[i] * travel_time[i, j] * z[j, i] for i in I for j in coverage[i]),
    GRB.MINIMIZE,
)

#solve it 
model.optimize()

chosen_locations_step2 = [j for j in J if x[j].X > 0.5]
covered_locations_step2 = [i for i in I if y[i].X > 0.5]

print("Step 2 selected locations:", chosen_locations_step2)
print("Step 2 covered demand:", sum(demand[i] for i in covered_locations_step2))
print("Step 2 objective:", model.ObjVal)

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.4.0 25E253)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 21332 rows, 21112 columns and 73675 nonzeros (Min)
Model fingerprint: 0x4af6f721
Model has 10122 linear objective coefficients
Variable types: 0 continuous, 21112 integer (21112 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [5e+02, 2e+07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 1e+06]

MIP start from previous solve produced solution with objective 5.05458e+08 (0.02s)
Loaded MIP start from previous solve with objective 5.05458e+08

Presolve removed 1 rows and 0 columns
Presolve time: 0.14s
Presolved: 21331 rows, 21112 columns, 73458 nonzeros
Variable types: 0 continuous, 21112 integer (21112 binary)

Root relaxation: cutoff, 15638 iterations, 0.35 seconds (0.81 work units)

    Nodes    |    Current Node    |     Objective Bou